In [ ]:
#| default_exp machine_learning.ml_augmentation

> **AI-generated — pending human review.** Created by Claude Code on 2026-10-22. Review checklist: `_nbdev_development/AI_REVIEW_CHECKLIST.md` (workspace root). Remove this cell only after review passes.

Historically augmentation ran *during* vault data extraction (`08_machine_learning_35` wraps `augment_text` around the HTML pieces while building datapoints), because there was no way to augment a finished dataset afterwards. R5 separates the two: pure datasets stay untouched, and augmentation becomes a named, versioned, regenerable transform over a pure dataset via `machine_learning.ml_data.augment_dataset`.

This module is that bridge: it wraps the string mutators of `helper.latex.augment` into row-level recipes registered in `AUGMENTATION_RECIPES`. Methods are tiered — *safe* ones vary only LaTeX styling (`\mathbf` vs `\mathbb`, `$` placement) and preserve labels; *noisy* ones change content (rename Greek letters, delete words/commands/keywords) and must be chosen per task. `push_dollar_signs` is deliberately excluded: it is self-declared buggy upstream.

In [ ]:
#| export
from typing import Callable

from trouver.helper.latex.augment import (augment_text, change_font_styles_at_random,
                                          change_greek_letters_at_random, dollar_sign_manipulation,
                                          random_char_modification, random_latex_command_removal,
                                          random_word_removal, remove_font_styles_at_random,
                                          remove_math_keywords)
from trouver.machine_learning.ml_data import AUGMENTATION_RECIPES

LATEX_SAFE_METHODS: list[tuple[Callable, float]] = [
    (remove_font_styles_at_random, 0.1),
    (change_font_styles_at_random, 0.1),
    (dollar_sign_manipulation, 0.1),
]
"""Label-preserving variation: only font styling and `$` placement change."""

LATEX_NOISY_METHODS: list[tuple[Callable, float]] = LATEX_SAFE_METHODS + [
    (change_greek_letters_at_random, 0.05),
    (random_char_modification, 0.02),
    (random_word_removal, 0.03),
    (random_latex_command_removal, 0.03),
    (remove_math_keywords, 0.1),
]
"""Safe methods plus content-changing noise; pick per task — keyword/word removal can invalidate labels."""

In [ ]:
#| export
def make_latex_recipe(column: str, methods: list[tuple[Callable, float]]) -> Callable:
    """
    Build an `ml_data` row recipe that applies all `methods` to `row[column]`.

    All other columns (labels, provenance) pass through untouched; every
    random choice comes from the recipe's own `rng`, so the augmented
    dataset is regenerable from (pure dataset, recipe id, seed).

    **Parameters**
    - `column` (*str*) — The row column holding the LaTeX text to mutate.
    - `methods` (*list[tuple[Callable, float]]*) — `(method, p)` pairs, e.g. `LATEX_SAFE_METHODS`.
    """
    def recipe(row: dict, rng) -> dict:
        funcs = [lambda x, m=m, p=p: m(x, p=p, rng=rng) for m, p in methods]
        row[column] = augment_text(row[column], funcs)
        return row
    return recipe


def register_latex_recipe(name: str, column: str,
                          methods: list[tuple[Callable, float]], version: int = 1) -> str:
    """
    Build a LaTeX recipe with `make_latex_recipe` and register it as `<name>_v<version>`.

    **Parameters**
    - `name` (*str*) — Recipe name, e.g. `'latex_robust'`.
    - `column` (*str*) — Row column to mutate.
    - `methods` (*list[tuple[Callable, float]]*) — `(method, p)` pairs.
    - `version` (*int*) — Bump when the method list or probabilities change.
    """
    recipe_id = f'{name}_v{version}'
    AUGMENTATION_RECIPES[recipe_id] = make_latex_recipe(column, methods)
    return recipe_id

In [ ]:
#| export
# Standard recipes for datasets whose text lives in a `text` column; tasks
# with other columns register their own via `register_latex_recipe`.
register_latex_recipe('latex_robust', 'text', LATEX_SAFE_METHODS)
register_latex_recipe('latex_noise', 'text', LATEX_NOISY_METHODS)

In [ ]:
import tempfile
from pathlib import Path

from fastcore.test import *

from trouver.machine_learning import ml_data as mld

In [ ]:
# The standard recipes are registered and flow through `augment_dataset`.
assert 'latex_robust_v1' in AUGMENTATION_RECIPES and 'latex_noise_v1' in AUGMENTATION_RECIPES

with tempfile.TemporaryDirectory() as tmp:
    root = Path(tmp)
    schema = mld.DatasetSchema('demo', 1, ('Note name', 'text'))
    pure = [{'Note name': 'a', 'text': r'$\mathbf{x} + \alpha$ is dense.'},
            {'Note name': 'b', 'text': r'Let $V$ be $\mathrm{span}(S)$. Definition 2.1.1 holds.'}]
    mld.save_rows(pure, mld.dataset_dir(root, schema) / 'demo.jsonl')

    out1 = mld.augment_dataset(root, schema, 'latex_robust_v1', seed=5)
    out2 = mld.augment_dataset(root, schema, 'latex_robust_v1', seed=5)
    test_eq(out1, out2)  # Regenerable through the real LaTeX mutators.
    test_eq([r['Note name'] for r in out1], ['a', 'b'])
    test_eq([r[mld.AUGMENTATION_COLUMN] for r in out1], ['latex_robust_v1'] * 2)

    # Noisy recipe runs end to end without crashing on mixed content.
    mld.augment_dataset(root, schema, 'latex_noise_v1', seed=5)

In [ ]:
# A forced-p recipe actually mutates the text column and nothing else.
register_latex_recipe('force_bold_removal', 'text', [(remove_font_styles_at_random, 1.0)])
with tempfile.TemporaryDirectory() as tmp:
    root = Path(tmp)
    schema = mld.DatasetSchema('demo', 1, ('Note name', 'text'))
    mld.save_rows([{'Note name': 'a', 'text': r'$\mathbf{x}$'}],
                  mld.dataset_dir(root, schema) / 'demo.jsonl')
    out = mld.augment_dataset(root, schema, 'force_bold_removal_v1', seed=0)
    test_eq(out[0]['text'], '$x$')

# `push_dollar_signs` is excluded from every standard recipe (upstream-buggy).
import trouver.helper.latex.augment as aug
for tier in (LATEX_SAFE_METHODS, LATEX_NOISY_METHODS):
    assert not any(m is aug.push_dollar_signs for m, _ in tier)

In [ ]:
#| hide
# Recipes must not read global random state: identical output under a
# perturbed global stream proves the row `rng` is the only source.
import random as _random
_recipe = make_latex_recipe('text', LATEX_SAFE_METHODS)
_row = {'Note name': 'a', 'text': r'$\mathbf{x} + \mathbb{y}$'}
_random.seed(12345)
a = _recipe(dict(_row), _random.Random(7))
_random.seed(999)
b = _recipe(dict(_row), _random.Random(7))
test_eq(a, b)

## Newline-stripping for `note_division`

Division rows join the left and right notes with `\n`, so in the canonical text a newline token sits at *every* positive seam — an input-visible cue that is absent at inference on an undivided note, and intra-note line wraps make newlines anti-correlated with seams besides. Replacing every `\n` with a space removes the cue from training; applying the same transform to notes at inference keeps train and test views identical. The replacement is one-for-one, so character offsets (the zero-width seam spans) stay valid untouched — no span remapping needed.

In [ ]:
#| export
def make_strip_newlines_recipe(column: str = 'text') -> Callable:
    """
    Build an `ml_data` row recipe replacing every newline in `row[column]` with a space.

    Length-preserving, so `spans` character offsets stay valid without remapping;
    deterministic (the `rng` is ignored), so the augmented dataset is trivially
    regenerable. Designed for `note_division` rows, whose `\n` joiner would
    otherwise mark every positive seam in the model input (see notebook text);
    apply the same replacement to note text at inference so views match.

    **Parameters**
    - `column` (*str*) — The row column holding the text to flatten.
    """
    def recipe(row: dict, rng) -> dict:
        row[column] = row[column].replace('\n', ' ')
        return row
    return recipe


def register_strip_newlines_recipe(name: str, column: str = 'text',
                                     version: int = 1) -> str:
    """
    Build a newline-strip recipe with `make_strip_newlines_recipe` and register it as `<name>_v<version>`.

    **Parameters**
    - `name` (*str*) — Recipe name, e.g. `'division_strip_newlines'`.
    - `column` (*str*) — Row column to flatten.
    - `version` (*int*) — Bump when the transform changes.
    """
    recipe_id = f'{name}_v{version}'
    AUGMENTATION_RECIPES[recipe_id] = make_strip_newlines_recipe(column)
    return recipe_id


In [ ]:
#| export
# The division task trains on the flattened view; eval stays pure (rule R5).
register_strip_newlines_recipe('division_strip_newlines')

In [ ]:
# Newline strip: length-preserving, spans stay valid, other keys untouched.
recipe = make_strip_newlines_recipe()
row = {'task': 'note_division', 'Note name': '_index_x',
       'text': 'ab\ncd\nef', 'spans': [{'start': 2, 'end': 2, 'label': 'split'}]}
out = recipe(dict(row), None)
test_eq(out['text'], 'ab cd ef')
test_eq(len(out['text']), len(row['text']))
test_eq(out['spans'], row['spans'])  # offsets unchanged, seam char still non-blank-left
test_eq(out['Note name'], '_index_x')


In [ ]:
# End-to-end through augment_dataset on real note_division schema rows.
from trouver.machine_learning.producers.note_division import (NOTE_DIVISION_SCHEMA,
                                                                save_note_division_dataset)
with tempfile.TemporaryDirectory() as td:
    root = Path(td)
    save_note_division_dataset(
        [{'task': 'note_division', 'Note name': '_index_x',
          'text': 'first note.\nsecond note.',
          'spans': [{'start': 12, 'end': 12, 'label': 'split'}]}], root)
    aug = mld.augment_dataset(root, NOTE_DIVISION_SCHEMA, 'division_strip_newlines_v1')
    test_eq(len(aug), 1)
    assert '\n' not in aug[0]['text']
    test_eq(aug[0]['text'], 'first note. second note.')
    test_eq(aug[0]['spans'][0]['start'], 12)
    test_eq(aug[0][mld.AUGMENTATION_COLUMN], 'division_strip_newlines_v1')
    test_eq(aug[0][mld.PROVENANCE_COLUMN], '_index_x')
